In [1]:
import pandas as pd
import datetime as dt

In [4]:
# 1. Load clean sales data
df = pd.read_csv(r"C:\Users\annus\cleaned_sales_data.csv", parse_dates=['order_date'])

# Set up a target baseline date (just after the latest transaction in your 2025 dataset
snapshot_date = df['order_date'].max() + dt.timedelta(days=1)

In [5]:
# 2. Group data by customer to compute Recency, Frequency, and Monetary Values
rfm = df.groupby('customer_id').agg({
    'order_date': lambda x: (snapshot_date-x.max()).days,
    'order_id': 'nunique',
    'total_sales': 'sum'
}).rename(columns={
    'order_date': 'Recency',
    'order_id': 'Frequency',
    'total_sales': 'Monetary'
})

In [6]:
# 3. Create scores from 1 to 4 using Quantiles
# Recency: Lower values (recent purchases) are better -> rank 4
rfm['R_score'] = pd.qcut(rfm['Recency'], 4, labels=[4, 3, 2, 1])

# Frequency & Monetary: Higher values are better -> rank 4
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4])
rfm['M_score'] = pd.qcut(rfm['Monetary'], 4, labels=[1, 2, 3, 4])

In [7]:
# 4. Synthesize into an overall Segment String
rfm['RFM_Cell'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)

In [9]:
# 5. Defime segments using simple business logic mapping
def assign_segment(row):
    r, f, m = int(row['R_score']), int(row['F_score']), int(row['M_score'])
    if r >= 3 and f >= 3:
        return 'Champions / Loyal'
    elif r <= 2 and f >= 3:
        return 'At Risk / Can\t Lose Them'
    elif r >= 3 and f <= 2:
        return 'Recent New Customer'
    else:
        return 'Hibernating / Dormat'

rfm['Segment'] = rfm.apply(assign_segment, axis=1)

# Save out the resulting segments table for dashboarding mapping
rfm.to_csv("customer_rfm_segments.csv")
print("RFM Customer Segments Exported Successfully!")

RFM Customer Segments Exported Successfully!


In [11]:
# ===================================================================================================================
# METHODOLOGY SELECTION
# ===================================================================================================================
"""
WHY SEGMENTATION ANALYSIS WAS CHOSEN:
1. Data Alignment: The transactional nature of 'cleaned_sales_data' contains perfect variables for RFM parameters 
   (customer identifiers, dates, and order values), whereas we lack web tracking traffic or subscription models for 
   funnel/cohort work.
2. Direct ROI: It maps numbers straight to marketing behavior. It distinguishes high-volume buyers from single-
   transaction users, optimizing future promotional spend.
"""

"\nWHY SEGMENTATION ANALYSIS WAS CHOSEN:\n1. Data Alignment: The transactional nature of 'cleaned_sales_data' contains perfect variables for RFM parameters \n   (customer identifiers, dates, and order values), whereas we lack web tracking traffic or subscription models for \n   funnel/cohort work.\n2. Direct ROI: It maps numbers straight to marketing behavior. It distinguishes high-volume buyers from single-\n   transaction users, optimizing future promotional spend.\n"

In [12]:
# ====================================================================================================================
# STRATEGIC BUSINESS VALUE & INSIGHTS SUMMARY
# ====================================================================================================================
"""
EXECUTIVE SUMMARY & MARKETING RECOMMENDATIONS:

Baes on the RFM Segmentation Analysis performed above on the 2025 transactional sales data, the customer base naturally 
fall into 4 behavioral tiers. Here are the strategic data-driven actions recommended for management:

1. CHAMPIONS / LOYAL CUSTOMERS
 - Insights: These individuals scored the highest across Recency and Frequency.
   They buy often, bought recently, and represent the highest total spend.
 - Business Action: Do not waste generic discount margins on them. Instead , enroll them into exclusive VIP referral 
   programs or provide early-bird access to upcoming product drops to leverage their brand advocacy.

2. AT RISK / CAN'T LOSE THEM
 - Insights: These customers have high historical Frequency and Monetary scores but a very low Recency score. They were 
   once highly valuable but have stopped purchasing recently.
 - Business Action: Immediate intervention required. Trigger targeted re-engagement email flows containing high-value coupon 
   incentive or "We Miss You" milestone promotions to win back their market share before they permanently churn.

3. RECENT NEW CUSTOMERS
 - Insights: They scored high on Recency but low on Frequency. These are buyers who have just discovered the store and placed
   their first order.
 - Business Action: Create a welocoming onboarding sequence. Send them educational content about related products alongside an
   automated secondary purchase discount valid for next 14 days to turn them into multi-tramsaction customers.

4. HIBERNATING / DORMAT 
 - Insights: Lowes metrics across the board (low Recency, Frequency, and Monetary).
             They bought a single low-value item months ago and never returned.
 - Business Action: Minimize expensive marketing spend here. Rely exclusively on low-budget, fully automated holiday broad-reach
   email campaigns. Focus capital on the 'At Risk' and 'New' segments instead.

""" 

'\nEXECUTIVE SUMMARY & MARKETING RECOMMENDATIONS:\n\nBaes on the RFM Segmentation Analysis performed above on the 2025 transactional sales data, the customer base naturally \nfall into 4 behavioral tiers. Here are the strategic data-driven actions recommended for management:\n\n1. CHAMPIONS / LOYAL CUSTOMERS\n - Insights: These individuals scored the highest across Recency and Frequency.\n   They buy often, bought recently, and represent the highest total spend.\n - Business Action: Do not waste generic discount margins on them. Instead , enroll them into exclusive VIP referral \n   programs or provide early-bird access to upcoming product drops to leverage their brand advocacy.\n\n2. AT RISK / CAN\'T LOSE THEM\n - Insights: These customers have high historical Frequency and Monetary scores but a very low Recency score. They were \n   once highly valuable but have stopped purchasing recently.\n - Business Action: Immediate intervention required. Trigger targeted re-engagement email flo